# Normalizing Flows

## Transformation of Random Variables
Before jumping into the model, it's useful to review the transformation of random variables and the intuition behind it. The core idea of transforming a random variable is that the probability mass must be conserved. Imagine squeezing a region of the input space into a smaller region of an output space, the density in that output space must be higher (imagine transforming a flat gaussian into a sharper one). Recall that for a random variable $X$, the transformation $Y=g(X)$ introduces the corresponding transformation in density:

$$f_Y(y)=f_X(g^{-1}(y)|\det J|)$$

where $|\det J|$ is the Jacobian determinant. I remember distinctly how in my introduction to probability class, this formula was given to us without much explanation. And without any explanation or intuition, this formula looks quite daunting, but in fact, it is not too hard to understand.

Taking inspiration from linear algebra, we know that the determinant represents a factor by which a transformation scales the area of volume. The Jacobian is the matrix of partial derivatives, which means it represents information of local change. Each entry basically answers: "if I nudge $x_i$ slightly, how much does $y_j$ move?" The determinant of the Jacobian, then, tells us how all of these local changes synthesize into one overarching net volume scaling factor. That is, if
- $|\det J| > 1$: the transformation **stretches** space (output region is bigger -> density goes down).
- $|\det J| < 1$: the transformation **compresses** space (output region is smaller -> density increases).
- $|\det J| = 1$: the transformation preserves volume (e.g. rotations) and density is unchanged.


## Model Setup
Let $x$ be a datapoint and let $f(z, \phi)$ be a transformation of a latent random variable $z\in \mathbb{R}^D$ with parameters $\phi$ and **tractable base distribution** $Pr(z)$. 

The main goal here is to transformation $x = f(z, \phi)$ where the parameters are chosen such that the resulting $Pr(x)$ contains the desired distribution. Drawing/generating new samples $x^*$, then, becomes a simple matter of drawing a sample $z^*$ and passing it through the function. From the above discussion, we know that the probability of $x$ under the transformed distribution is:
$$Pr(x \mid \phi) = \left| \frac{\partial f(z, \phi)}{\partial z} \right|^{-1} \cdot Pr(z)$$
where the first term is the inverse of the $D\times D$ absolute Jacobian determinant.

To draw the sample, we need the **forward mapping/generative direction** $x=f(z, \phi)$. We also need the **backward mapping/normalizing direction** $z=f^{-1}(x, \phi)$ in order to learn the parameters $\phi$. As such, we need to choose the $f$ judiciously so that it's invertible. Since the base density $Pr(z)$ is generally chosen to be a standard Gaussian, the backward mapping is often termed the **normalizing direction**, turning a complex distribution over $x$ and normalizing it to $z$. Learning here is quite straightforward and involves finding parameters $\phi$ that maximizes likelihood of the training data and therefore minimizes the negative log likelihood:
$$
\begin{align*}
\hat{\phi} &= \arg\max_\phi \left[\prod^N_{i=1}Pr(x_i\mid \phi) \right] \\
&= \arg\min_\phi \left[ \sum^N_{i=1} \log\left[\left|\frac{\partial f(z_i, \phi)}{\partial z_i} \right| \right] - \log\left[ Pr(z_i)\right]\right]
\end{align*}
$$

## Mapping with neural networks

In practice, the forward mapping is usually defined by a neural network consisting of a series of layers $f_k(\cdot, \phi_k)$ composed together as:
$$x = f(z, \phi) = f_K\left(f_{K-1}\left( ... f_1(z, \phi_1)..., \phi_{K-1}\right), \phi_K\right)$$
and the backward direction would be
$$z = f^{-1}(x, \phi) = f^{-1}_1\left(f_2^{-1}(... f^{-1}_K(x, \phi_K) ..., \phi_2), \phi_1\right)$$

Since the base density is usually defined as a standard Gaussian, each layer in the backward mapping has the effect of gradually normalizing the distribution over $x$. In other words, it **flows** toward the normal. 

Additionally, the Jacobian of the density can be expressed as:

$$\left|\frac{\partial f(z, \phi)}{\partial z}\right| = \left|\frac{\partial f_K(f_{K-1}, \phi_K)}{\partial f_{K-1}} \right| \cdot \left|\frac{\partial f_{K-1}(f_{K-2}, \phi)}{\partial f_{K-2}} \right| \cdot ... \cdot \left|\frac{\partial f_2(f_1, \phi)}{\partial f_1} \right| \cdot \left|\frac{\partial f_1(z, \phi)}{\partial z} \right|$$

where the notation $f_k$ is overloaded to represent the output of the $k$-th neural network layer. The chain structure we see here is a result of the multivariate chain rule. Recall that $\det (AB) = \det A \cdot \det B$. Thus, the determinant chains across all layers of the network. Intuitively, we can think of each determinant as the scaling factor on the already distorted space of the previous transformation. As such, the chained determinants have a multiplicative effect across the entire transformation composition.

We see that the idea behind normalizing flows is somewhat straightforward, however, there are some properties that the network must adhere to:
1.  Collectively, the set of network layers must be suﬀiciently expressive to map a multivariate standard normal distribution to an arbitrary density.
2. The network layers must be invertible; each must define a unique one-to-one mapping from any input point to an output point (a bijection). If multiple inputs were mapped to the same output, the inverse would be ambiguous.
3. It must be possible to compute the inverse of each layer eﬀiciently. We need to do this every time we evaluate the likelihood. This happens repeatedly during training, so there must be a closed-form solution or a fast algorithm for the inverse.
4. It also must be possible to evaluate the determinant of the Jacobian eﬀiciently for either the forward or inverse mapping.

## Invertible network layers


Let's now take a quick tour of invertible network layers, or **flows**, and talk about the tradeoffs.

### Linear flows
A **linear flow** has the form $f(h)=\beta + \Omega h$, where $\Omega$ is an invertible matrix. For $\Omega \in \mathbb{R}^{D\times D}$, the inverse computation and determinant computation are both $O(D^3)$. There are ways to improve the performance by using special matrices (e.g. diagonal or triangular), but this reduces tradeoffs in expressiveness.

Unfortunately, linear flows are inherently limited in their expressiveness. This is because a linear transformation of a Gaussian is still Gaussian. That is, if we apply $f$ to a normally distributed variable with $N(\mu, \Sigma)$, then the result will also be normally distributed with $N(\mu + \Omega\beta, \Omega \Sigma \Omega^\top)$. As such, it's not possible to map any variable to an arbitrary non-Gaussian density.

### Element-wise flows
Moving to our first non-linear flow, we have **element-wise flows**. The idea here is to apply a simple point-wise non-linear function $f(\dot, \phi)$ to each element of the input:
$$f(h) = \left[f(h_1, \phi), \dots, f(h_D, \phi)\right]^\top$$
The resulting Jacobian $\frac{\partial f(h)}{h}$ has a diagonal structure since the $d$-th input onlt affects the $d$-th output:
$$
J = \frac{\partial f(h)}{\partial h}
=
\begin{bmatrix}
\frac{\partial f(h_1)}{\partial h_1} & 0 & \cdots & 0 \\
0 & \frac{\partial f(h_2)}{\partial h_2} & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & \frac{\partial f(h_D)}{\partial h_D}
\end{bmatrix}
$$
Thus, the determinant is the product of all the diagonal entries:
$$\left|\frac{\partial f(h)}{\partial h}\right| = \prod_{d=1}^D \left|\frac{\partial f(h_d)}{\partial h_d}\right|$$

The is a decent amount of choice when it comes to choosing $f$. One path is to use a fixed invertible, non-linear function such as leaky ReLU. In this case, there are no parameters required. Another choice is to use a piecewise linear function with $K$ regions, such as
$$f(h_d, \phi) = (\sum^{b-1}_{i=1} \phi_i) + (hK - b)\phi_b$$
Here, the function maps $[0,1] \rightarrow [0,1]$ and $\phi_1,...,\phi_K$ are positive, summing to $K$. Furthermore, $b=\lfloor hK \rfloor$ is the floor function, corresponding to the bin indices. The first term in the equation above sums all the preceding bins and the second term represents the proportion of the way through the $b$-th bin that $h$ lies. This function is easy to invert, and its gradient can be calculated almost everywhere. There are many similar schemes for creating smooth functions, often using splines with parameters that ensure the function is monotonic and hence invertible.

Element-wise flows are non-linear but a major limitation is that there are no correlations between dimensions. Suppose that $f(x_1) = z_1$ and $f(x_2)=z_2$. It follows that $x_1$ cannot depend on $z_2$ and $x_2$ cannot depend on $z_1$, which means that the transformation can't create interactions between the two dimensions.

### Coupling flows
**Coupling flows** divide the input into two parts $h=[h_1^\top, h_2^\top]^\top$ and the flow is defined as
$$
\begin{align*}
h'_1 &= h_1 \\
h'_2 &= g(h_2, \phi(h_1))
\end{align*}
$$

where $g(\dot, \phi)$ are element-wise flows (or some other appropriate flow) and $\phi$ are themselves non-linear functions of the input $h_1$. This, set up is a little bit confusing so let's spell each part out clearly. The first component of the input is mapped to the output of the layer exactly as is. The second component of the input is a function of both the first and second input components. The canonical version of this is an affine transformation where
$$g(h_2, \phi(h_1)) = h_2 \odot \exp(s(h_1)) + t(h_1)$$
where $s(\cdot)$ is a scale neural network and $t(\cdot)$ is a translation network. We see that, unlike elementwise flows, $h'_2$ is now **conditioned on** $h_1$ in addition to $h_2$, which accounts for relationships across the dimensions. So what about the Jacobian calculation? It turns out this set up makes the Jacobian calculation mcuh more efficient. In particular, we have
$$
J = \frac{\partial h'}{\partial h}
=
\begin{array}{c}
\\[-1.5ex]
\begin{array}{cc}
h_1 & \qquad h_2
\end{array}
\\
\begin{array}{c}
h'_1 \\
h'_2
\end{array}
\!\!
\begin{bmatrix}
I_d & 0 \\
\frac{\partial h'_2}{\partial h_1}
&
\operatorname{diag}\!\left(
\exp\left[s(h_1)\right]
\right)
\end{bmatrix}
\end{array}
$$

Since the first component mapping $h'_1 = h_1$, that part of the Jacobian (top left) is the identity $I_d$. Furthermore, $h'_1$ does not interact with $h_2$, which means that part of the Jacobian (top right) is $0$. $h'_2$ relates to $h_1$ through the neural networks $s$ and $t$, as reflected on the bottom left block. Finally, the $h'_2$ is an affine transformation of $h_2$, the partial comes out to $\operatorname{diag}\left(\exp\left[s(h_1)\right]\right)$. This means that we have a lower triangular matrix and the determinant of a lower triangular matrix 

$$
\det \begin{bmatrix}
A & 0 \\
B & C
\end{bmatrix} = \det(A) \det(C)
$$

Voila! While the lower left block results in a complex partial derivative that must take into account the neural network, it contributes nothing to the determinant.

The inverse and Jacobian can be computed eﬀiciently, but this approach only transforms the second half of the parameters in a way that depends on the first half. To make a more general transformation, the elements of $h$ are randomly shuffled using permutation matrices between layers, so every variable is ultimately transformed by every other.